In [ ]:
!pip -q install "transformers>=4.44.2" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.1" datasets pillow pandas torch torchvision --upgrade

!pip install git+https://github.com/huggingface/transformers

In [ ]:
# ============================================================
# GPU 메모리 상태 확인
# ============================================================
import torch

print("🔍 현재 GPU 메모리 상태:")
print(f"  할당됨: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"  예약됨: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"  최대 사용: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")
print(f"  전체 용량: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
print(f"  여유 공간: {free_memory / 1024**3:.2f} GB")

if torch.cuda.memory_allocated(0) / 1024**3 > 35:
    print("\n⚠️ 경고: GPU 메모리가 거의 가득 찼습니다!")
    print("   이것이 느린 이유일 수 있습니다.")


# 셀 1: 라이브러리 임포트 및 기본 설정

In [ ]:
import os, math, random, re
from datetime import datetime
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)

# ---------- 성능 관련 환경 변수 / 백엔드 설정 ----------
# 메모리 단편화 줄이고 큰 텐서 할당 스톨 방지
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

# 토크나이저 병렬 경고 및 fork 이슈 방지
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# CUDA / cudnn 성능 최적화
torch.backends.cudnn.benchmark = True

# TF32 허용 → A100 Tensor Core 경로 활용 (FP32 연산 가속)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# AMP dtype 통일 (A100에서 안정적이고 빠름)
AMP_DTYPE = torch.bfloat16

# 이미지 로드 시 픽셀 제한 해제 (아주 큰 이미지 대비)
Image.MAX_IMAGE_PIXELS = None

# 디바이스 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


# 셀 2: 학습 설정 (여기만 수정)

In [ ]:
# 모델 설정
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
IMAGE_SIZE = 224  # 224로 유지 (속도 우선). 더 점수 필요하면 256/336로 올리기.
SEED = 42

# 하이퍼파라미터 (속도/성능 타협 반영)
BATCH_SIZE = 8        # GPU에 한 번에 더 크게 태움
GRAD_ACCUM = 2        # 누적 단계를 줄여서 루프 오버헤드 감소. Effective batch ~= 16
EPOCHS = 1            # 1 -> 2 로 늘려서 점수 향상 기대
LEARNING_RATE = 5e-5  # 1e-4에서 낮춤, 과적합/폭주 줄이기
WEIGHT_DECAY = 0.01
NUM_WORKERS = 8       # DataLoader 멀티프로세싱 ↑, GPU 유휴시간 ↓
NUM_TRAIN_SAMPLES = 3887  # None이면 전체 사용

# 저장 경로
VERSION_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
BASE_SAVE_DIR = "/home/team036/content"
MODEL_SAVE_DIR = f"{BASE_SAVE_DIR}/Qwen3-VL-4B-finetuned_{VERSION_NAME}"

# 데이터 경로
TRAIN_CSV = "/home/team036/aichallenge/train.csv"
TEST_CSV  = "/home/team036/aichallenge/test.csv"

print(f"\n{'='*60}")
print(f"📦 Model Version: {VERSION_NAME}")
print(f"💾 Save Directory: {MODEL_SAVE_DIR}")
print(f"{'='*60}\n")

# 재현성 고정
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 셀 3: 데이터 로드 및 분할

In [ ]:

print("Loading data...")
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# 선택적으로 학습 샘플 수 제한
if NUM_TRAIN_SAMPLES is not None:
    train_df = train_df.sample(n=NUM_TRAIN_SAMPLES, random_state=SEED).reset_index(drop=True)
    print(f"✓ Sampled {NUM_TRAIN_SAMPLES} samples for training")
else:
    print(f"✓ Using all {len(train_df)} samples for training")

# Train/Valid split (85:15)
split_idx = int(len(train_df) * 0.85)
train_subset = train_df.iloc[:split_idx].reset_index(drop=True)
valid_subset = train_df.iloc[split_idx:].reset_index(drop=True)

print(f"✓ Train samples (pre-balance): {len(train_subset)}")
print(f"✓ Valid samples: {len(valid_subset)}")
print(f"✓ Test samples: {len(test_df)}")

# ---- 클래스 불균형 보정 (oversampling) ----
# 목표: 각 정답('a','b','c','d')의 빈도를 가장 많은 클래스 수준까지 맞춰서 모델이 한 글자만 반복 출력하는 걸 방지
if "answer" in train_subset.columns:
    counts = train_subset["answer"].value_counts()
    max_count = counts.max()
    balanced_chunks = []
    for ans_label, cnt in counts.items():
        subset_label = train_subset[train_subset["answer"].astype(str).str.strip().str.lower() == str(ans_label).strip().lower()]
        if cnt < max_count:
            # 복제해서 oversampling
            extra = subset_label.sample(max_count - cnt, replace=True, random_state=SEED)
            balanced_chunks.append(pd.concat([subset_label, extra], ignore_index=True))
        else:
            balanced_chunks.append(subset_label)
    train_subset_balanced = pd.concat(balanced_chunks, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print("✓ Oversampling applied for class balance")
else:
    train_subset_balanced = train_subset
    print("⚠️ No 'answer' column for balancing; using original train subset")

print(f"✓ Train samples (post-balance): {len(train_subset_balanced)}")


# 셀 4: 프롬프트 생성 함수

In [ ]:
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "You must answer using exactly one lowercase letter from {a,b,c,d}. "
    "No explanation, no punctuation."
)

def clean_text(x: str):
    x = str(x)
    x = x.replace("\t", " ").replace("\n", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x

def build_mc_prompt(question, a, b, c, d):
    # 노이즈 제거
    question = clean_text(question)
    a = clean_text(a)
    b = clean_text(b)
    c = clean_text(c)
    d = clean_text(d)

    # 답 형식을 강하게 강제
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Rules:\n"
        "1. Think silently.\n"
        "2. Then answer ONLY one character from {a,b,c,d}.\n"
        "3. Output format: a or b or c or d.\n"
        "4. Do NOT include any explanation or punctuation."
    )

# 셀 5: 모델 및 프로세서 로드

In [ ]:
print("\nLoading model and processor...")

# 양자화 설정: 4bit + bf16 연산으로 A100 Tensor Core 활용
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # AMP_DTYPE와 일관
)

# 프로세서 로드 (우리가 직접 224로 resize하므로 min/max_pixels 강제 안 함)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# 베이스 모델 로드
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",            # 단일 A100이면 전체 cuda:0에 들어갈 것
    trust_remote_code=True,
)

# k-bit 학습 준비 (LoRA 대상 파라미터 grad 켜주고 norm 안정화 등)
base_model = prepare_model_for_kbit_training(base_model)

# gradient checkpointing은 속도를 느리게 하므로, VRAM 여유 있으면 끈다
# (중요: Qwen 계열은 내부에서 checkpointing 켜고 use_cache=False로 바꾸는 경우가 있다.
#  여기서 다시 명시적으로 끄고 cache를 돌려놓는다.)
if hasattr(base_model, "gradient_checkpointing_disable"):
    base_model.gradient_checkpointing_disable()
if hasattr(base_model, "config"):
    base_model.config.use_cache = True  # forward 속도 향상

print("✓ Base model loaded & checkpointing disabled / use_cache restored")

# 셀 6: LoRA 설정 및 PEFT 모델 생성

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("\n🔍 LoRA 적용 확인:")
lora_modules = [name for name, _ in model.named_modules() if 'lora' in name.lower()]
if lora_modules:
    print(f"✅ LoRA가 적용된 모듈 수: {len(lora_modules)}")
    print(f"   예시: {lora_modules[0]}")
else:
    print("⚠️  경고: LoRA가 적용되지 않았습니다! target_modules를 확인하세요.")

model = model.to(device)


# 셀 7: Dataset 및 Collator 정의

In [ ]:
class VQAMCDataset(Dataset):
    """
    - 이미지를 즉시 224x224 (또는 IMAGE_SIZE)로 리사이즈 → CPU 전처리 비용 최소화
    - prompt는 '한 글자만 답해' 규칙을 강제
    - answer는 반드시 'a'/'b'/'c'/'d' 한 글자
    """
    def __init__(self, df, image_size=224):
        self.df = df.reset_index(drop=True)
        self.image_size = image_size
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img = Image.open(row["path"]).convert("RGB")
        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)

        user_text = build_mc_prompt(
            row["question"],
            row["a"], row["b"], row["c"], row["d"]
        )
        answer = str(row["answer"]).strip().lower()

        # 정답은 한 글자만 유지
        if len(answer) > 1:
            answer = answer[0]

        return {
            "image": img,
            "user_text": user_text,
            "answer": answer,
        }

class VQACollator:
    """
    - 각 샘플을 멀티모달 chat 템플릿으로 변환
    - processor로 batch tensor화
    - labels = input_ids 복제 (teacher forcing)
    """
    def __init__(self, processor, system_instruct):
        self.processor = processor
        self.system_instruct = system_instruct
    
    def __call__(self, batch):
        images = []
        texts = []

        for sample in batch:
            img = sample["image"]
            user_text = sample["user_text"]
            answer = sample["answer"]

            messages = [
                {
                    "role": "system",
                    "content": [{"type": "text", "text": self.system_instruct}],
                },
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": img},
                        {"type": "text",  "text": user_text},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [{"type": "text", "text": answer}],
                },
            ]

            # Qwen3-VL chat template → 단일 text 시퀀스로 serialize
            text = self.processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )

            images.append(img)
            texts.append(text)
        
        encoded = self.processor(
            text=texts,
            images=images,
            return_tensors="pt",
            padding=True,
        )

        # LM 학습: labels = input_ids 복제
        encoded["labels"] = encoded["input_ids"].clone()
        return encoded

print("✓ Dataset and Collator defined")


# 셀 8: DataLoader 생성

In [ ]:

train_dataset = VQAMCDataset(train_subset_balanced, image_size=IMAGE_SIZE)
valid_dataset = VQAMCDataset(valid_subset,          image_size=IMAGE_SIZE)

collate_fn = VQACollator(processor, SYSTEM_INSTRUCT)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    prefetch_factor=4,
    persistent_workers=True,
)

print("✓ DataLoaders created")

# 셀 9: Optimizer 및 Scheduler 설정

In [ ]:

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8,
)

# 실제 optimizer.step()은 GRAD_ACCUM마다 1번 발생하므로, 그 기준으로 step 수 계산
update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = EPOCHS * update_steps_per_epoch
warmup_steps = int(num_training_steps * 0.03)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=num_training_steps,
)

# bf16에서는 GradScaler 필요 없음 (fp16용 오버헤드 제거)
scaler = torch.cuda.amp.GradScaler(enabled=False)

print(f"\n{'='*60}")
print(f"Training Configuration")
print(f"  • Epochs: {EPOCHS}")
print(f"  • Batch size: {BATCH_SIZE}")
print(f"  • Gradient accumulation: {GRAD_ACCUM}")
print(f"  • Effective batch size: ~{BATCH_SIZE * GRAD_ACCUM}")
print(f"  • Learning rate: {LEARNING_RATE}")
print(f"  • Weight decay: {WEIGHT_DECAY}")
print(f"  • Update steps/epoch: {update_steps_per_epoch}")
print(f"  • Total training steps: {num_training_steps}")
print(f"  • Warmup steps: {warmup_steps}")
print(f"  • NUM_WORKERS: {NUM_WORKERS}")
print(f"  • Gradient checkpointing: DISABLED (speed priority)")
print(f"  • AMP_DTYPE: {AMP_DTYPE}")
print(f"{'='*60}\n")

# 디버깅: 병목 지점 찾기

In [ ]:
import time

print("\n🔍 병목 지점 테스트...")

# 1. 데이터 로딩 속도
start = time.time()
batch = next(iter(train_loader))
data_time = time.time() - start
print(f"  데이터 로딩: {data_time:.2f}초")

# 2. GPU 전송 속도
start = time.time()
batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
gpu_time = time.time() - start
print(f"  GPU 전송: {gpu_time:.2f}초")

# 3. Forward pass 속도
start = time.time()
with torch.amp.autocast('cuda', dtype=torch.bfloat16):
    outputs = model(**batch)
forward_time = time.time() - start
print(f"  Forward pass: {forward_time:.2f}초")

# 4. Backward pass 속도
start = time.time()
loss = outputs.loss / GRAD_ACCUM
scaler.scale(loss).backward()
backward_time = time.time() - start
print(f"  Backward pass: {backward_time:.2f}초")

print(f"\n  총 시간: {data_time + gpu_time + forward_time + backward_time:.2f}초")
print(f"  병목: ", end="")
times = {"데이터 로딩": data_time, "GPU 전송": gpu_time, 
         "Forward": forward_time, "Backward": backward_time}
print(max(times, key=times.get))

optimizer.zero_grad(set_to_none=True)
print()

# 셀 9.5: GPU 메모리 정리

In [ ]:
import gc

print("🧹 GPU 메모리 정리 중...")

# 1. Python 가비지 컬렉션
gc.collect()

# 2. CUDA 캐시 정리
torch.cuda.empty_cache()

# 3. 메모리 상태 확인
print(f"GPU 메모리 상태:")
print(f"  할당됨: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"  예약됨: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"  최대 사용: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f} GB")

# 4. 메모리 초기화
torch.cuda.reset_peak_memory_stats()
torch.cuda.reset_accumulated_memory_stats()

print("✓ 메모리 정리 완료\n")

# 셀 10: 학습 루프 실행

In [ ]:
global_step = 0
best_val_acc = 0.0
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # ---------------- Training ----------------
    model.train()
    train_loss_accum = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for step, batch in enumerate(progress_bar, start=1):
        # GPU로 이동
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # bf16 autocast (Tensor Core 경로)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        train_loss_accum += loss.item()

        # 그래디언트 누적 후 실제 step
        if step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = train_loss_accum / GRAD_ACCUM
            progress_bar.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}"
            })
            train_loss_accum = 0.0

    # 남은 누적 그라디언트 처리 (에폭 마지막에 step % GRAD_ACCUM != 0일 수 있음)
    remainder = (step % GRAD_ACCUM)
    if remainder != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        global_step += 1

    # ---------------- Validation ----------------
    model.eval()
    val_loss_total = 0.0
    val_steps = 0
    correct = 0
    total = 0

    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
        for batch in tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]"):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            outputs = model(**batch)

            # loss
            val_loss_total += outputs.loss.item()
            val_steps += 1

            # accuracy 계산:
            # 각 샘플마다 마지막 유효 라벨 토큰(= -100이 아닌 마지막 토큰) 위치를 찾아서
            # 그 위치에서 logits argmax와 라벨이 일치하면 correct++
            logits = outputs.logits         # (B, T, V)
            labels = batch["labels"]        # (B, T)

            B = labels.size(0)
            for i in range(B):
                label_ids = labels[i]  # (T,)
                valid_pos = torch.where(label_ids != -100)[0]
                if valid_pos.numel() == 0:
                    continue
                last_pos = valid_pos[-1]
                gold_id = label_ids[last_pos].item()
                pred_id = torch.argmax(logits[i, last_pos], dim=-1).item()

                if gold_id == pred_id:
                    correct += 1
                total += 1

    avg_val_loss = val_loss_total / max(1, val_steps)
    val_acc = correct / total if total > 0 else 0.0

    print(f"\n[Epoch {epoch+1}] Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # ---------------- Best 모델 저장 ----------------
    # 기본적으로 accuracy를 우선 기준으로 저장
    save_flag = False
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_flag = True
    # 혹은 acc가 같으면 loss로 비교해서 더 낮은 loss 저장
    elif (abs(val_acc - best_val_acc) < 1e-6) and (avg_val_loss < best_val_loss):
        save_flag = True

    if save_flag:
        best_val_loss = avg_val_loss
        print(f"  ✓ New best model! Saving to {MODEL_SAVE_DIR}")

        os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

        # LoRA 어댑터만 저장됨 (full base model은 안 덤프)
        model.save_pretrained(MODEL_SAVE_DIR)
        processor.save_pretrained(MODEL_SAVE_DIR)

        with open(f"{MODEL_SAVE_DIR}/training_info.txt", "w") as f:
            f.write(f"Model Version: {VERSION_NAME}\n")
            f.write(f"Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Base Model: {MODEL_ID}\n")
            f.write(f"Epoch: {epoch+1}/{EPOCHS}\n")
            f.write(f"Validation Loss: {avg_val_loss:.4f}\n")
            f.write(f"Validation Acc: {val_acc:.4f}\n")
            f.write(f"Batch Size: {BATCH_SIZE}\n")
            f.write(f"Gradient Accumulation: {GRAD_ACCUM}\n")
            f.write(f"Learning Rate: {LEARNING_RATE}\n")
            f.write(f"Weight Decay: {WEIGHT_DECAY}\n")
            f.write(f"Train Samples (balanced): {len(train_subset_balanced)}\n")
            f.write(f"Valid Samples: {len(valid_subset)}\n")


print(f"\n{'='*60}")
print(f"🎉 Training Complete!")
print(f"  • Best Validation Acc: {best_val_acc:.4f}")
print(f"  • Best Validation Loss: {best_val_loss:.4f}")
print(f"  • Model Version: {VERSION_NAME}")
print(f"  • Saved at: {MODEL_SAVE_DIR}")
print(f"{'='*60}\n")

# GPU 메모리 정리
del model, optimizer, scheduler
torch.cuda.empty_cache()
print("✓ GPU memory cleared")

# 문제 생길시

## 1 gradient checkpointing은 지금 OFF 상태로 주석 처리

그대로 두고 돌려보고 VRAM 여유 있으면 keep.
만약 OOM 뜨면 다시 켜주기

In [ ]:
base_model.gradient_checkpointing_enable()